# LGN Converter — Kaggle URL Stream Test

Before running: open **Notebook options**, set **Accelerator = GPU** and **Internet = On**. This notebook verifies source access before loading Whisper.

## 1. Configure the authorized test range

In [ ]:
SOURCE_URL = "https://www.youtube.com/watch?v=4tk4uI5BzTw"
SOURCE_LANGUAGE = "zh"
CLIP_START_SECONDS = 0
CLIP_DURATION_SECONDS = 30  # Use 30 first; increase only after it succeeds.

assert 0 < CLIP_DURATION_SECONDS <= 180
print("Configuration ready")


## 2. Verify GPU and install the project

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import torch

assert torch.cuda.is_available(), "Enable a GPU in Kaggle Notebook options"
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

repo_dir = Path("/kaggle/working/lngconverter")
if (repo_dir / ".git").exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/cubatic/lngconverter.git", str(repo_dir)], check=True)

os.chdir(repo_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[inference]"], check=True)
subprocess.run([sys.executable, "-c", "import pydantic_settings, faster_whisper, lgnconverter"], check=True)
commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("Repository commit:", commit)
print("Installation complete")


## 3. Verify that Kaggle can access the source
This step does not download the video or load Whisper.

In [ ]:
from lgnconverter.config import Settings
from lgnconverter.source import resolve_source

settings = Settings(enable_youtube_source=True)
source = resolve_source(SOURCE_URL, settings)
print("Title:", source.title)
print("Source duration:", source.duration)
print("Source type:", source.kind)
assert source.duration and source.duration > CLIP_DURATION_SECONDS
print("Source access successful")


## 4. Extract the bounded range and transcribe

In [ ]:
import time

from lgnconverter.jobs import JobStore
from lgnconverter.models import CreateJobRequest

settings = Settings(
    enable_youtube_source=True,
    whisper_model="small",
    whisper_device="cuda",
    whisper_compute_type="float16",
    max_source_seconds=180,
    decode_timeout_seconds=900,
    keep_audio_artifacts=False,
)
store = JobStore(settings)
job = store.create(CreateJobRequest(
    source_url=SOURCE_URL,
    authorization_confirmed=True,
    source_language=SOURCE_LANGUAGE,
    target_language="hi",
    clip_start_seconds=CLIP_START_SECONDS,
    clip_duration_seconds=CLIP_DURATION_SECONDS,
))
print("Job ID:", job.id)

last_status = None
while True:
    result = store.get(job.id)
    if result.status != last_status:
        print("Status:", result.status.value)
        last_status = result.status
    if result.status.value in {"completed", "failed"}:
        break
    time.sleep(3)

if result.status.value == "failed":
    raise RuntimeError(result.error)

print("Detected language:", result.detected_language)
print(f"Processed duration: {result.duration_seconds:.2f}s")
print("Segments:", len(result.transcript))
for segment in result.transcript:
    print(f"[{segment.start:7.2f} → {segment.end:7.2f}] {segment.text}")
